# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khaled-dragon/ML-intern/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
%pip -q install duckdb huggingface_hub
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

Paste your HF READ token (hf_...): ··········


In [2]:
signal1 = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']} WHERE month='2026-03'),
    prior AS (
        SELECT content_hash_id,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS imp_prev30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.month = '2026-03'
        GROUP BY 1
    ),
    outcome AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS imp_next30
        FROM {TABLES['fact_daily']} WHERE month = '2026-04'
        GROUP BY 1
    )
    SELECT
        CASE WHEN p.imp_prev30 < 100 THEN 'low_volume'
             WHEN p.imp_prev30 < 1000 THEN 'mid_volume'
             ELSE 'high_volume' END AS volume_bucket,
        COUNT(*) AS n,
        AVG(CASE WHEN o.imp_next30 > 1.2 * p.imp_prev30 THEN 1 ELSE 0 END) AS growth_rate
    FROM prior p JOIN outcome o USING (content_hash_id)
    WHERE p.imp_prev30 > 0
    GROUP BY 1
    ORDER BY 2 DESC
""").df()
print(signal1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  volume_bucket      n  growth_rate
0    low_volume  75419     0.327716
1    mid_volume  56460     0.263638
2   high_volume  44389     0.246705


In [3]:
signal2 = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']} WHERE month='2026-03'),
    prior AS (
        SELECT content_hash_id,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS imp_prev30,
               AVG(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_avg_position END) AS pos_prev30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.month = '2026-03'
        GROUP BY 1
    ),
    outcome AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS imp_next30
        FROM {TABLES['fact_daily']} WHERE month = '2026-04'
        GROUP BY 1
    )
    SELECT
        CASE WHEN p.pos_prev30 <= 10 THEN 'top_10'
             WHEN p.pos_prev30 <= 30 THEN 'top_30'
             ELSE 'beyond_30' END AS position_bucket,
        COUNT(*) AS n,
        AVG(CASE WHEN o.imp_next30 > 1.2 * p.imp_prev30 THEN 1 ELSE 0 END) AS growth_rate
    FROM prior p JOIN outcome o USING (content_hash_id)
    WHERE p.imp_prev30 >= 100
    GROUP BY 1
    ORDER BY 2 DESC
""").df()
print(signal2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_bucket      n  growth_rate
0          top_10  55402     0.269395
1          top_30  32801     0.239444
2       beyond_30  12646     0.241737


**Signal 1 — Volume (linked to the quick-win flag): OPPOSITE.**
Low-volume pages had the highest growth rate (32.8%) and high-volume pages
the lowest (24.7%) — the reverse of what a "high volume = quick win" rule
would assume. This is likely partly a measurement artifact: a 20% growth
threshold is easier to cross for pages with small baseline numbers, where
normal noise can swing the percentage a lot. Either way, using raw volume
as a positive signal for growth would be misleading, so I won't use it
directly as a "higher is better" input.

**Signal 2 — Position: MIXED.**
Growth rates were close across all position buckets (26.9%, 23.9%, 24.2%),
with no clear monotonic pattern. Position alone isn't a strong differentiator
for future growth in this window, so I'll treat it as a secondary filter
rather than a primary scoring signal.

**Rule design decision:** given both signals, my rule will score pages
based on a mid-range volume band with confirmed traffic (not near-zero, to
avoid noise) combined with reasonable position, and flag pages already
showing early momentum, rather than assuming "more volume/better position
always means more upside."

### "####################################################################"

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
rule_data = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']} WHERE month='2026-03'),
    prior AS (
        SELECT content_hash_id, client_hash_id,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS imp_prev30,
               AVG(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_avg_position END) AS pos_prev30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.month = '2026-03'
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM prior
""").df()

# The rule: mid-range volume (not tiny, not huge) + decent position = higher score
def score_row(imp, pos):
    volume_score = 1 - abs(imp - 500) / 5000  # peaks near 500 impressions
    position_score = max(0, (50 - pos) / 50) if pos <= 50 else 0
    return round(0.5 * volume_score + 0.5 * position_score, 3)

rule_data['score'] = rule_data.apply(lambda r: score_row(r['imp_prev30'], r['pos_prev30']), axis=1)
rule_data['reason_code'] = 'mid_volume_decent_position'
rule_data['action'] = 'expand_or_promote'

queue = rule_data.sort_values('score', ascending=False).reset_index(drop=True)

import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(queue):,} rows")
queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote 100,849 rows


,content_hash_id,client_hash_id,imp_prev30,pos_prev30,score,reason_code,action
0,content_2b7d06046c20f975,client_0fa64a184f18a4a0,473.0,0.217777,0.995,mid_volume_decent_position,expand_or_promote
1,content_baf11dfb6980c0c2,client_e547b89c05043229,550.0,0.029000,0.995,mid_volume_decent_position,expand_or_promote
2,content_80ac249c600ec136,client_e547b89c05043229,494.0,0.504467,0.994,mid_volume_decent_position,expand_or_promote
3,content_4248bf9a32883531,client_73cda7b4e4f265ea,521.0,0.516417,0.993,mid_volume_decent_position,expand_or_promote
4,content_72fe5b2dfc61d6b3,client_23a62021009f63c4,493.0,0.853853,0.991,mid_volume_decent_position,expand_or_promote
5,content_8bed25de40e03d20,client_e547b89c05043229,542.0,0.510428,0.991,mid_volume_decent_position,expand_or_promote
6,content_75d11fc01b374310,client_62f4a7e64f5e0096,529.0,0.722472,0.990,mid_volume_decent_position,expand_or_promote
7,content_8bab091bd3580a96,client_73cda7b4e4f265ea,534.0,0.663369,0.990,mid_volume_decent_position,expand_or_promote
8,content_d333eedf3670ace3,client_e547b89c05043229,455.0,0.627395,0.989,mid_volume_decent_position,expand_or_promote
9,content_1e98c7b61c32f90f,client_62f4a7e64f5e0096,549.0,0.612274,0.989,mid_volume_decent_position,expand_or_promote


In [5]:
con.sql(f"""
    SELECT MIN(gsc_avg_position), MAX(gsc_avg_position), AVG(gsc_avg_position)
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03' AND gsc_avg_position IS NOT NULL
""").df()

,min(gsc_avg_position),max(gsc_avg_position),avg(gsc_avg_position)
0,0.0,498.0,15.826651


### "####################################################################"

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
top10 = queue.head(10).copy()

top10['why_here'] = top10.apply(
    lambda r: f"imp_prev30={r['imp_prev30']:.0f} (near the mid-volume sweet spot) "
              f"and pos_prev30={r['pos_prev30']:.2f} (very strong position)", axis=1)

top10['what_would_make_it_wrong'] = top10['pos_prev30'].apply(
    lambda p: "This may be a 'position zero' / featured-snippet page already at its "
              "ceiling — promoting it further may have little real upside, the rule "
              "can't distinguish 'already winning' from 'room to grow'."
    if p < 2 else
    "If the mid-volume number reflects a temporary traffic dip rather than a stable "
    "pattern, the rule would misread noise as opportunity."
)

top10[['content_hash_id', 'action', 'reason_code', 'why_here', 'what_would_make_it_wrong']]

,content_hash_id,action,reason_code,why_here,what_would_make_it_wrong
0,content_2b7d06046c20f975,expand_or_promote,mid_volume_decent_position,imp_prev30=473 (near the mid-volume sweet spot...,This may be a 'position zero' / featured-snipp...
1,content_baf11dfb6980c0c2,expand_or_promote,mid_volume_decent_position,imp_prev30=550 (near the mid-volume sweet spot...,This may be a 'position zero' / featured-snipp...
2,content_80ac249c600ec136,expand_or_promote,mid_volume_decent_position,imp_prev30=494 (near the mid-volume sweet spot...,This may be a 'position zero' / featured-snipp...
3,content_4248bf9a32883531,expand_or_promote,mid_volume_decent_position,imp_prev30=521 (near the mid-volume sweet spot...,This may be a 'position zero' / featured-snipp...
4,content_72fe5b2dfc61d6b3,expand_or_promote,mid_volume_decent_position,imp_prev30=493 (near the mid-volume sweet spot...,This may be a 'position zero' / featured-snipp...
5,content_8bed25de40e03d20,expand_or_promote,mid_volume_decent_position,imp_prev30=542 (near the mid-volume sweet spot...,This may be a 'position zero' / featured-snipp...
6,content_75d11fc01b374310,expand_or_promote,mid_volume_decent_position,imp_prev30=529 (near the mid-volume sweet spot...,This may be a 'position zero' / featured-snipp...
7,content_8bab091bd3580a96,expand_or_promote,mid_volume_decent_position,imp_prev30=534 (near the mid-volume sweet spot...,This may be a 'position zero' / featured-snipp...
8,content_d333eedf3670ace3,expand_or_promote,mid_volume_decent_position,imp_prev30=455 (near the mid-volume sweet spot...,This may be a 'position zero' / featured-snipp...
9,content_1e98c7b61c32f90f,expand_or_promote,mid_volume_decent_position,imp_prev30=549 (near the mid-volume sweet spot...,This may be a 'position zero' / featured-snipp...


### "####################################################################"

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
# Leakage check: confirm the score formula only used prior-window inputs
print("Columns used in score formula:", ['imp_prev30', 'pos_prev30'])
print("Both are aggregated strictly from report_date within month=2026-03")
print("(the prior/feature window). No column from month=2026-04 (the outcome")
print("window used only in the earlier signal tests) was used in scoring.")
print()
print(f"Rows where pos_prev30 < 2 (likely position-zero pages): "
      f"{(queue.head(10)['pos_prev30'] < 2).sum()} out of top 10")

Columns used in score formula: ['imp_prev30', 'pos_prev30']
Both are aggregated strictly from report_date within month=2026-03
(the prior/feature window). No column from month=2026-04 (the outcome
window used only in the earlier signal tests) was used in scoring.

Rows where pos_prev30 < 2 (likely position-zero pages): 10 out of top 10


**Weak picks:** All 10 of the top-10 rows share the same problem: they're
all "position zero" pages (pos_prev30 near 0), which the rule scores highest
because of how the position_score term is built. But these pages are
already at their ceiling, arguably not the ones with real room to grow.
A better rule would exclude or down-weight pages already at position zero
and instead surface pages with strong-but-improvable positions (say,
position 5-15) paired with healthy volume, that's where genuine
"expand or promote" opportunity actually lives. I'm keeping this version
as the honest baseline and noting the fix as a clear target for the
Week-5 model to beat.

**Leakage check:** the score only uses imp_prev30 and pos_prev30, both
computed strictly from the prior (feature) window within month=2026-03.
No column from the outcome window (month=2026-04) or any label-derived
value was used in the rule or the score. The imp_next30 column used
earlier (in the signal tests) never entered the scoring formula.

Looking closer at the top-10, most had pos_prev30 near 0, which likely
represents "position zero" (featured snippet) pages, not a data error
(the full column ranges 0-498 with a mean of ~15.8). The rule's position
component over-rewards these already-elite pages instead of surfacing
genuine mid-tier opportunities. This is a weak pick pattern: the rule
confuses "already winning" with "worth promoting."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.